In [1]:
# Import Libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# READ DATA - AS FILE WASN'T ENCODED SO USED ENCODING
df = pd.read_csv(
    r'C:\Users\Admin\Desktop\BIA Capstone Project\data\raw\data.csv', encoding='cp1252')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [4]:
# Describe
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


In [5]:
# DATASET DESCRIPTION

# Quantity - Number of qty purchased by the customer
# Invoice date - Purchase date
# Unit Price - Per Unit price
# Customer ID - Unique ID given to the customer

In [6]:
df.shape

(541909, 8)

In [7]:
# Check null values
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [8]:
# Check data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [9]:
# Check duplicate values
df.duplicated().sum()

np.int64(5268)

STEP 2: DATA CLEANING

In [10]:
# Converting Invoice date to Datetime datatype
df['InvoiceDate'] =pd.to_datetime(df['InvoiceDate'])

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [12]:
# Dropped CustomerID where value is Null because rows with missing CustomerID cannot be associated with any specific customer
df = df.dropna(subset=['CustomerID'])

In [13]:
df.shape

(406829, 8)

In [14]:
# Drop duplicate values
df.drop_duplicates(inplace=True)

In [15]:
# Check if price is negative
(df['UnitPrice'] <=0).sum()


np.int64(40)

In [16]:
df = df[df['UnitPrice'] > 0]

In [17]:
df.shape

(401564, 8)

In [18]:
# Create two different Datasets One is for valid purchases and one for returns/cancellations
valid_df = df[
    (~df['InvoiceNo'].astype(str).str.startswith('C')) &
    (df['Quantity']>0) &
    (df['UnitPrice']>0)
].copy()
valid_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [19]:
valid_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 392692 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    392692 non-null  object        
 1   StockCode    392692 non-null  object        
 2   Description  392692 non-null  object        
 3   Quantity     392692 non-null  int64         
 4   InvoiceDate  392692 non-null  datetime64[ns]
 5   UnitPrice    392692 non-null  float64       
 6   CustomerID   392692 non-null  float64       
 7   Country      392692 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 27.0+ MB


In [20]:
valid_df.shape

(392692, 8)

In [21]:
# Create returns dataset
return_df = df[
    (df['InvoiceNo'].astype(str).str.startswith('C')) |
    (df['Quantity'] < 0)
].copy()
return_df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


In [22]:
return_df.shape[0]

8872

In [23]:
# Customers with most returns
return_df.groupby('CustomerID')['Quantity'].sum().sort_values()

CustomerID
16446.0   -80995
12346.0   -74215
15838.0    -9361
15749.0    -9014
16029.0    -6476
           ...  
16566.0       -1
14853.0       -1
13316.0       -1
16618.0       -1
14934.0       -1
Name: Quantity, Length: 1589, dtype: int64

In [24]:
# Countries with highest return
return_df.groupby('Country')['Quantity'].sum().sort_values()

Country
United Kingdom       -259167
EIRE                   -4196
Germany                -1815
France                 -1623
USA                    -1424
Spain                  -1127
Netherlands             -809
Japan                   -798
Australia               -556
Sweden                  -446
Switzerland             -305
Italy                   -113
Norway                   -91
Belgium                  -85
Czech Republic           -79
Portugal                 -78
Israel                   -56
Austria                  -54
Denmark                  -47
Cyprus                   -44
Finland                  -38
Poland                   -31
Malta                    -26
Channel Islands          -12
Singapore                 -7
Saudi Arabia              -5
European Community        -2
Greece                    -1
Name: Quantity, dtype: int64

In [25]:
valid_df.to_csv('valid_purchases.csv', index=False)

In [26]:
return_df.to_csv('order_returns.csv', index=False)